# Материалы учебного проекта

Пути адаптированы к структуре репозитория; пустые ячейки удалены. Расчётная логика сохранена. Исторические выводы могут отражать промежуточные запуски. Перепроверенные показатели находятся в README, ограничения — в supporting/methodology_notes.md.


In [3]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/faucets_sales_2023_2024.xlsx').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the extracted repository folder.')
RAW_FILE = PROJECT_ROOT / 'data/raw/faucets_sales_2023_2024.xlsx'
DATA_OUTPUT = PROJECT_ROOT / 'data/processed'
FIGURE_OUTPUT = PROJECT_ROOT / 'reports/generated'
DATA_OUTPUT.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT.mkdir(parents=True, exist_ok=True)

"""
=============================================================
ПРОГНОЗ ВЫРУЧКИ РЫНКА КУХОННЫХ СМЕСИТЕЛЕЙ НА 2025 ГОД
Модель: Holt-Winters (Exponential Triple Smoothing)
Данные: Январь 2023 — Декабрь 2024 (24 точки)
=============================================================
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ─────────────────────────────────────────────────────────────
# 1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ
# ─────────────────────────────────────────────────────────────

df = pd.read_excel(RAW_FILE)
df['month'] = pd.to_datetime(df['month'])
df = df[(df['revenue'] > 0) & (df['quantity'] > 0)]

monthly = (df.groupby('month')['revenue']
             .sum()
             .reset_index()
             .sort_values('month'))

ts = monthly.set_index('month')['revenue'] / 1e6
ts.index = pd.DatetimeIndex(ts.index).to_period('M').to_timestamp()
ts.index.freq = 'MS'

print("=== ИСХОДНЫЙ РЯД ===")
print(f"Период:      {ts.index[0].strftime('%Y-%m')} — {ts.index[-1].strftime('%Y-%m')}")
print(f"Точек:       {len(ts)}")
print(f"Мин:         {ts.min():.1f} млн руб.")
print(f"Макс:        {ts.max():.1f} млн руб.")
print(f"Среднее:     {ts.mean():.1f} млн руб.")
print("\nЗначения по месяцам:")
for dt, v in ts.items():
    print(f"  {dt.strftime('%Y-%m')}: {v:>7.1f}")

# ─────────────────────────────────────────────────────────────
# 2. ДЕКОМПОЗИЦИЯ
# ─────────────────────────────────────────────────────────────

decomp = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
axes[0].plot(decomp.observed,  color='#2C3E50', lw=2)
axes[1].plot(decomp.trend,     color='#9ECD58', lw=2)
axes[2].plot(decomp.seasonal,  color='#A2C6ED', lw=2)
axes[3].plot(decomp.resid,     color='#E8A838', lw=1.5)

axes[0].set_title('Исходный ряд',  fontsize=11, fontweight='bold')
axes[1].set_title('Тренд',         fontsize=11, fontweight='bold')
axes[2].set_title('Сезонность',    fontsize=11, fontweight='bold')
axes[3].set_title('Остатки',       fontsize=11, fontweight='bold')
for ax in axes:
    ax.set_ylabel('млн руб.')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Декомпозиция временного ряда (STL)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURE_OUTPUT / 'decomposition.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✅ Сохранено: decomposition.png")

print("\n=== СЕЗОННЫЕ ИНДЕКСЫ ===")
monthly_mean = ts.mean()
mn = ['Янв','Фев','Мар','Апр','Май','Июн','Июл','Авг','Сен','Окт','Ноя','Дек']
for i in range(12):
    v23 = ts.iloc[i]
    v24 = ts.iloc[i + 12]
    idx = ((v23 + v24) / 2) / monthly_mean
    status = 'пик ↑' if idx > 1.1 else ('спад ↓' if idx < 0.9 else 'норма')
    print(f"  {mn[i]}: {idx:.2f}  ({status})")

# ─────────────────────────────────────────────────────────────
# 3. МОДЕЛЬ HOLT-WINTERS (основная)
# ─────────────────────────────────────────────────────────────

print("\n=== МОДЕЛЬ HOLT-WINTERS ===")
model_hw = ExponentialSmoothing(
    ts,
    trend='add',
    seasonal='add',
    seasonal_periods=12,
    initialization_method='estimated'
)
fit_hw = model_hw.fit(optimized=True)

print(f"α (уровень): {fit_hw.params['smoothing_level']:.4f}")
print(f"β (тренд):   {fit_hw.params['smoothing_trend']:.4f}")
print(f"γ (сезон):   {fit_hw.params['smoothing_seasonal']:.4f}")

forecast_hw = fit_hw.forecast(12)
forecast_hw.index = pd.date_range('2025-01', periods=12, freq='MS')

resid_std = fit_hw.resid.std()
ci95_lo = forecast_hw - 1.96 * resid_std
ci95_hi = forecast_hw + 1.96 * resid_std
ci68_lo = forecast_hw - resid_std
ci68_hi = forecast_hw + resid_std

print("\nПрогноз 2025 (млн руб.):")
print(f"{'Месяц':<10} {'Прогноз':>10} {'CI 95% нижн.':>15} {'CI 95% верх.':>15}")
print("-" * 52)
for dt, v, lo, hi in zip(forecast_hw.index, forecast_hw, ci95_lo, ci95_hi):
    print(f"{dt.strftime('%Y-%m'):<10} {v:>10.1f} {lo:>15.1f} {hi:>15.1f}")

print(f"\nИтого 2025:           {forecast_hw.sum():>10.1f} млн руб.")
print(f"2024 факт:            {ts.iloc[12:].sum():>10.1f} млн руб.")
print(f"YoY рост:             {(forecast_hw.sum() - ts.iloc[12:].sum()) / ts.iloc[12:].sum() * 100:>+10.1f}%")
print(f"CI 95% год (нижн.):   {ci95_lo.sum():>10.1f} млн руб.")
print(f"CI 95% год (верхн.):  {ci95_hi.sum():>10.1f} млн руб.")

# ─────────────────────────────────────────────────────────────
# 4. АЛЬТЕРНАТИВА — SARIMA
# ─────────────────────────────────────────────────────────────

print("\n=== АЛЬТЕРНАТИВА: SARIMA(1,1,1)(1,1,1)[12] ===")
try:
    sarima = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12),
                     enforce_stationarity=False, enforce_invertibility=False)
    fit_sarima = sarima.fit(disp=False)
    fc_sarima = fit_sarima.get_forecast(12)
    fc_sarima_mean = fc_sarima.predicted_mean
    print(f"Итого SARIMA 2025: {fc_sarima_mean.sum():.1f} млн руб.")
    print(f"Holt-Winters:      {forecast_hw.sum():.1f} млн руб.")
    delta = abs(forecast_hw.sum() - fc_sarima_mean.sum())
    print(f"Расхождение:       {delta:.1f} млн руб.  ({'модели согласуются ✅' if delta < 200 else 'расхождение значимо ⚠️'})")
except Exception as e:
    print(f"SARIMA не сошлась: {e}")
    fc_sarima_mean = None

# ─────────────────────────────────────────────────────────────
# 5. БЭКТЕСТ (ручная инициализация, т.к. train < 2*12)
# ─────────────────────────────────────────────────────────────

print("\n=== БЭКТЕСТ (обучение 18 мес., тест 6 мес.) ===")
train = ts.iloc[:18]   # янв 2023 — июн 2024
test  = ts.iloc[18:]   # июл — дек 2024

# ----- Ручной расчёт начальных компонентов по первым 12 точкам train -----
first_cycle = train.iloc[:12].values   # полный сезонный цикл
init_level = np.mean(first_cycle)
init_trend = (first_cycle[-1] - first_cycle[0]) / 11.0   # прирост за 11 шагов
init_seasonal = first_cycle / init_level
# Нормализация, чтобы сумма индексов = 12
init_seasonal = init_seasonal / np.sum(init_seasonal) * 12

model_test = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=12,
    initialization_method='known',
    initial_level=init_level,
    initial_trend=init_trend,
    initial_seasonal=init_seasonal
)
fit_test = model_test.fit(optimized=True)
pred_test = fit_test.forecast(6)
pred_test.index = test.index

print(f"{'Месяц':<10} {'Факт':>8} {'Прогноз':>10} {'Ошибка':>8} {'Ошибка%':>9}")
print("-" * 48)
for dt, fact, pred in zip(test.index, test.values, pred_test.values):
    err = pred - fact
    err_pct = err / fact * 100
    print(f"{dt.strftime('%Y-%m'):<10} {fact:>8.1f} {pred:>10.1f} {err:>+8.1f} {err_pct:>+8.1f}%")

mae  = np.mean(np.abs(pred_test.values - test.values))
mape = np.mean(np.abs((pred_test.values - test.values) / test.values)) * 100
rmse = np.sqrt(np.mean((pred_test.values - test.values) ** 2))

print(f"\nMAE:  {mae:.1f} млн руб.")
print(f"MAPE: {mape:.1f}%  ({'отлично' if mape < 5 else ('хорошо' if mape < 10 else 'приемлемо')})")
print(f"RMSE: {rmse:.1f} млн руб.")

# ─────────────────────────────────────────────────────────────
# 6. ИТОГОВЫЙ ГРАФИК
# ─────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ts.index, ts.values, color='#2C3E50', lw=2.5, marker='o', ms=5, label='Факт 2023–2024', zorder=5)
ax.plot(fit_hw.fittedvalues.index, fit_hw.fittedvalues.values, color='#A2C6ED', lw=1.5, ls='--', alpha=0.8, label='Модель (fitted)')
ax.plot(forecast_hw.index, forecast_hw.values, color='#9ECD58', lw=2.5, marker='o', ms=5, label='Прогноз 2025 (Holt-Winters)', zorder=5)
ax.fill_between(forecast_hw.index, ci95_lo, ci95_hi, alpha=0.12, color='#9ECD58', label='CI 95%')
ax.fill_between(forecast_hw.index, ci68_lo, ci68_hi, alpha=0.22, color='#9ECD58', label='CI 68%')
if fc_sarima_mean is not None:
    fc_sarima_mean.index = forecast_hw.index
    ax.plot(fc_sarima_mean.index, fc_sarima_mean.values, color='#E8A838', lw=1.8, ls=':', marker='s', ms=4, label='SARIMA (альтернатива)', alpha=0.85)

ax.axvline(ts.index[-1], color='#BDBDBD', lw=1.5, ls='--')
ax.text(ts.index[-1], ax.get_ylim()[1] * 0.96, '  прогноз →', fontsize=10, color='#999')
ax.annotate(f"{forecast_hw.sum():.0f} млн\n+{(forecast_hw.sum() - ts.iloc[12:].sum()) / ts.iloc[12:].sum() * 100:.1f}% YoY",
            xy=(forecast_hw.index[-1], forecast_hw.iloc[-1]), xytext=(-90, 20), textcoords='offset points',
            fontsize=10, fontweight='bold', color='#4a7a10', arrowprops=dict(arrowstyle='->', color='#9ECD58', lw=1.5))

ax.set_ylabel('Выручка, млн руб.', fontsize=12)
ax.set_title(f'Прогноз выручки рынка кухонных смесителей на 2025 год\nМодель: Holt-Winters (ETS AAA) · бэктест MAPE = {mape:.1f}%',
             fontsize=13, fontweight='bold', color='#2C3E50')
ax.legend(loc='upper left', fontsize=10)
ax.grid(axis='y', alpha=0.25)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIGURE_OUTPUT / 'forecast_2025.png', dpi=150, bbox_inches='tight')
plt.close()
print("\n✅ Сохранено: forecast_2025.png")

# ─────────────────────────────────────────────────────────────
# 7. СВОДКА
# ─────────────────────────────────────────────────────────────

print("\n" + "=" * 52)
print("ИТОГОВАЯ СВОДКА ПРОГНОЗА")
print("=" * 52)
print(f"Метод:          Holt-Winters (ETS AAA)")
print(f"Обучение:       24 точки (Янв 2023 — Дек 2024)")
print(f"Горизонт:       12 месяцев (2025)")
print(f"MAPE бэктест:   {mape:.1f}%")
print(f"Прогноз 2025:   {forecast_hw.sum():.0f} млн руб.")
print(f"YoY к 2024:     +{(forecast_hw.sum() - ts.iloc[12:].sum()) / ts.iloc[12:].sum() * 100:.1f}%")
print(f"CI 95% (год):   [{ci95_lo.sum():.0f}, {ci95_hi.sum():.0f}] млн руб.")
print(f"Файлы:          decomposition.png, forecast_2025.png")
print("=" * 52)

=== ИСХОДНЫЙ РЯД ===
Период:      2023-01 — 2024-12
Точек:       24
Мин:         103.2 млн руб.
Макс:        330.3 млн руб.
Среднее:     243.3 млн руб.

Значения по месяцам:
  2023-01:   103.2
  2023-02:   135.1
  2023-03:   173.0
  2023-04:   176.7
  2023-05:   180.8
  2023-06:   188.1
  2023-07:   217.5
  2023-08:   237.2
  2023-09:   237.1
  2023-10:   264.3
  2023-11:   315.4
  2023-12:   324.0
  2024-01:   251.7
  2024-02:   271.0
  2024-03:   284.5
  2024-04:   249.5
  2024-05:   242.9
  2024-06:   236.6
  2024-07:   265.7
  2024-08:   271.0
  2024-09:   277.6
  2024-10:   300.9
  2024-11:   304.4
  2024-12:   330.3

✅ Сохранено: decomposition.png

=== СЕЗОННЫЕ ИНДЕКСЫ ===
  Янв: 0.73  (спад ↓)
  Фев: 0.83  (спад ↓)
  Мар: 0.94  (норма)
  Апр: 0.88  (спад ↓)
  Май: 0.87  (спад ↓)
  Июн: 0.87  (спад ↓)
  Июл: 0.99  (норма)
  Авг: 1.04  (норма)
  Сен: 1.06  (норма)
  Окт: 1.16  (пик ↑)
  Ноя: 1.27  (пик ↑)
  Дек: 1.34  (пик ↑)

=== МОДЕЛЬ HOLT-WINTERS ===
α (уровень): 1.0000
β (тре